# TP 2 — Explorer et préparer un jeu de transactions
## Séance 2 : Données de transactions, exploration et préparation
### Version corrigée (enseignant)

| | |
|---|---|
| **École** | ESAIP |
| **Classe** | Aix IR 5 CP |
| **Cours** | Détection d’anomalies et fraude avec Python |
| **Enseignant** | Serigne Faye |

## Contexte

Votre banque vous confie un fichier de **10 040 transactions** par carte enregistrées sur trois mois (janvier à mars 2026). Une petite partie est frauduleuse. Avant toute modélisation, vous devez :

1. charger le fichier et le comprendre ;
2. repérer et corriger les problèmes de qualité ;
3. explorer le lien entre les variables et la fraude ;
4. préparer les données pour un futur modèle, **sans fuite de données**.

**Rappel des colonnes**

| Colonne | Contenu |
|---|---|
| `id_transaction`, `id_client` | identifiants |
| `date_heure` | date et heure de la transaction |
| `montant` | montant en euros |
| `categorie` | type de commerce |
| `canal` | carte présente, sans contact ou en ligne |
| `pays` | pays de la transaction |
| `fraude` | 1 = fraude, 0 = transaction normale (**cible**) |

## Étape 1 — Création du fichier

Exécutez la cellule suivante **sans la modifier**. Elle crée le fichier `transactions.csv` dans le dossier du notebook. Vous allez ensuite le charger comme s’il vous avait été fourni par la banque.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def generer_transactions(seed=7):
    rng = np.random.default_rng(seed)
    n = 10000
    n_clients = 500

    poids = np.array([1] * 6 + [4] * 16 + [2] * 2, dtype=float)
    poids /= poids.sum()

    categories = ['alimentation', 'restauration', 'transport', 'vetements',
                  'electronique', 'voyage', 'jeux_en_ligne']
    p_cat = [0.30, 0.22, 0.15, 0.12, 0.08, 0.06, 0.07]
    mu = {'alimentation': 3.3, 'restauration': 3.2, 'transport': 2.8, 'vetements': 4.0,
          'electronique': 5.0, 'voyage': 5.3, 'jeux_en_ligne': 3.0}

    categorie = rng.choice(categories, size=n, p=p_cat)
    montant = np.array([rng.lognormal(mu[c], 0.5) for c in categorie])
    heure = rng.choice(24, size=n, p=poids)
    canal = rng.choice(['carte_presente', 'sans_contact', 'en_ligne'], size=n, p=[0.45, 0.30, 0.25])
    canal[categorie == 'jeux_en_ligne'] = 'en_ligne'
    pays = rng.choice(['FR', 'ES', 'IT', 'DE', 'US', 'NG'], size=n, p=[0.90, 0.03, 0.03, 0.02, 0.015, 0.005])
    fraude = np.zeros(n, dtype=int)

    # 100 fraudes : la nuit, en ligne, à l'étranger, montants élevés
    idx = rng.choice(n, size=100, replace=False)
    fraude[idx] = 1
    heure[idx] = np.where(rng.random(100) < 0.8, rng.integers(0, 6, 100), rng.integers(6, 24, 100))
    categorie[idx] = rng.choice(['electronique', 'voyage', 'jeux_en_ligne'], size=100, p=[0.4, 0.3, 0.3])
    montant[idx] = rng.lognormal(5.2, 0.6, 100)
    canal[idx] = np.where(rng.random(100) < 0.9, 'en_ligne', 'carte_presente')
    pays[idx] = np.where(rng.random(100) < 0.6, rng.choice(['US', 'NG', 'ES', 'DE'], size=100), 'FR')

    jour = rng.integers(0, 90, n)
    minute = rng.integers(0, 60, n)
    date_heure = (pd.Timestamp('2026-01-01') + pd.to_timedelta(jour, unit='D')
                  + pd.to_timedelta(heure, unit='h') + pd.to_timedelta(minute, unit='m'))

    df = pd.DataFrame({
        'id_transaction': [f'T{i:05d}' for i in range(n)],
        'id_client': [f'C{c:03d}' for c in rng.integers(0, n_clients, n)],
        'date_heure': date_heure,
        'montant': montant.round(2),
        'categorie': categorie,
        'canal': canal,
        'pays': pays,
        'fraude': fraude,
    })

    # Achats exceptionnels légitimes (anomalies réelles, mais pas des fraudes)
    legit = rng.choice(df.index[df['fraude'] == 0], size=3, replace=False)
    df.loc[legit, 'montant'] = [3200.00, 4500.00, 5800.00]
    df.loc[legit, 'categorie'] = 'voyage'

    # Problèmes de qualité : erreurs de saisie, valeurs négatives
    non_fraude = df.index[df['fraude'] == 0].to_numpy()
    saisie = rng.choice(non_fraude, size=4, replace=False)
    df.loc[saisie, 'montant'] = 99999.99
    neg = rng.choice(np.setdiff1d(non_fraude, saisie), size=6, replace=False)
    df.loc[neg, 'montant'] = -df.loc[neg, 'montant']

    # Valeurs manquantes
    df.loc[rng.choice(n, size=100, replace=False), 'montant'] = np.nan
    df.loc[rng.choice(n, size=200, replace=False), 'canal'] = np.nan

    # Doublons exacts
    doublons = df.sample(40, random_state=1)
    df = pd.concat([df, doublons], ignore_index=True)
    df = df.sort_values('date_heure', kind='stable').reset_index(drop=True)
    return df

# Création du fichier transactions.csv dans le dossier du notebook
generer_transactions().to_csv('transactions.csv', index=False)
print('Fichier transactions.csv créé.')

## Étape 2 — Chargement et premier regard

Complétez les cellules (remplacez chaque `...`). Pensez à convertir la colonne `date_heure` en vraie date lors de la lecture.

In [ ]:
df = pd.read_csv('transactions.csv', parse_dates=['date_heure'])
print(df.shape)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## Étape 3 — Contrôle de la qualité des données

Cherchez les trois types de problèmes vus en cours : valeurs manquantes, doublons, valeurs aberrantes.

In [ ]:
# 1. Valeurs manquantes : nombre et pourcentage par colonne
manquantes = df.isna().sum()
print(manquantes)
print((manquantes / len(df) * 100).round(2))

# 2. Doublons exacts
nb_doublons = df.duplicated().sum()
print('Doublons :', nb_doublons)

# 3. Montants négatifs ou nuls
nb_negatifs = (df['montant'] <= 0).sum()
print('Montants <= 0 :', nb_negatifs)

# 4. Les 10 plus grands montants (colonnes montant, categorie, fraude)
df.nlargest(10, 'montant')[['montant', 'categorie', 'fraude']]

## Étape 4 — Nettoyage

Règles à appliquer, dans cet ordre :

1. supprimer les doublons exacts ;
2. considérer comme **erreurs** les montants négatifs ou nuls et les montants supérieurs ou égaux à 99 999 € (code de saisie) : les remplacer par une valeur manquante ;
3. remplacer les montants manquants par la **médiane de la catégorie** ;
4. remplacer les canaux manquants par `'inconnu'`.

Les achats de plusieurs milliers d’euros qui restent plausibles ne sont **pas** des erreurs : on les garde.

In [ ]:
# 1. Supprimer les doublons
df = df.drop_duplicates()

# 2. Montants négatifs ou nuls, ou >= 99999 : remplacés par une valeur manquante
masque_erreur = (df['montant'] <= 0) | (df['montant'] >= 99999)
df.loc[masque_erreur, 'montant'] = np.nan

# 3. Montants manquants : médiane de la catégorie
df['montant'] = df['montant'].fillna(df.groupby('categorie')['montant'].transform('median'))

# 4. Canal manquant : 'inconnu'
df['canal'] = df['canal'].fillna('inconnu')

# Vérification
print(df.shape)
print('Valeurs manquantes restantes :', df.isna().sum().sum())

## Étape 5 — Exploration : quelles variables sont liées à la fraude ?

In [ ]:
# Taux de fraude global
taux_global = df['fraude'].mean()
print(f'Taux de fraude global : {taux_global:.2%}')

# Moyenne et médiane du montant
print('Montant moyen :', round(df['montant'].mean(), 1))
print('Montant médian :', round(df['montant'].median(), 1))

# Variables dérivées
df['heure'] = df['date_heure'].dt.hour
df['nuit'] = (df['heure'] < 6).astype(int)
df['etranger'] = (df['pays'] != 'FR').astype(int)

**Taux de fraude par catégorie**

In [ ]:
# Taux de fraude (moyenne de la colonne fraude) par groupe, du plus élevé au plus faible
df.groupby('categorie')['fraude'].agg(['mean', 'sum', 'count']).round(4).sort_values('mean', ascending=False)

**Taux de fraude par canal**

In [ ]:
df.groupby('canal')['fraude'].agg(['mean', 'sum', 'count']).round(4).sort_values('mean', ascending=False)

**Taux de fraude par pays**

In [ ]:
df.groupby('pays')['fraude'].agg(['mean', 'sum', 'count']).round(4).sort_values('mean', ascending=False)

**Taux de fraude selon la nuit (heure < 6)**

In [ ]:
df.groupby('nuit')['fraude'].agg(['mean', 'sum', 'count']).round(4).sort_values('mean', ascending=False)

**Taux de fraude selon le pays (étranger ou non)**

In [ ]:
df.groupby('etranger')['fraude'].agg(['mean', 'sum', 'count']).round(4).sort_values('mean', ascending=False)

## Étape 6 — Visualisation

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))

# Histogramme du montant en échelle linéaire
ax[0].hist(df['montant'], bins=60)

# Histogramme du montant en échelle logarithmique
bins = np.logspace(np.log10(df['montant'].min()), np.log10(df['montant'].max()), 50)
ax[1].hist(df['montant'], bins=bins)
ax[1].set_xscale('log')

ax[0].set_title('Échelle linéaire')
ax[1].set_title('Échelle logarithmique')
plt.show()

In [ ]:
# Diagramme en barres du taux de fraude (en %) selon l'heure
taux_heure = df.groupby('heure')['fraude'].mean() * 100
plt.figure(figsize=(10, 3.2))
plt.bar(taux_heure.index, taux_heure.values)
plt.xlabel('Heure')
plt.ylabel('Taux de fraude (%)')
plt.xticks(range(24))
plt.show()

In [ ]:
# Boîtes à moustaches du montant selon la classe (fraude 0 / 1), échelle logarithmique
plt.figure(figsize=(5, 4))
plt.boxplot([df.loc[df['fraude'] == 0, 'montant'], df.loc[df['fraude'] == 1, 'montant']])
plt.xticks([1, 2], ['Normale (0)', 'Fraude (1)'])
plt.yscale('log')
plt.ylabel('Montant (€)')
plt.show()

## Étape 7 — Préparation des variables sans fuite de données

On découpe **dans le temps** : entraînement = transactions avant le 2 mars 2026, test = à partir du 2 mars 2026. Toutes les transformations apprises (moyenne, écart-type) sont calculées sur l’entraînement uniquement.

In [ ]:
# 1. Découpage temporel
train = df[df['date_heure'] < '2026-03-02'].copy()
test = df[df['date_heure'] >= '2026-03-02'].copy()
print('Entraînement :', len(train), 'lignes,', train['fraude'].sum(), 'fraudes')
print('Test         :', len(test), 'lignes,', test['fraude'].sum(), 'fraudes')

In [ ]:
# 2. Standardisation du montant : paramètres calculés sur l'ENTRAÎNEMENT uniquement
mu = train['montant'].mean()
sigma = train['montant'].std()
train['montant_std'] = (train['montant'] - mu) / sigma
test['montant_std'] = (test['montant'] - mu) / sigma    # mêmes mu et sigma !
print(f'mu = {mu:.2f}   sigma = {sigma:.2f}')
print('Entraînement : moyenne =', round(train['montant_std'].mean(), 3), ' écart-type =', round(train['montant_std'].std(), 3))
print('Test         : moyenne =', round(test['montant_std'].mean(), 3), ' écart-type =', round(test['montant_std'].std(), 3))

In [ ]:
# 3. Encodage one-hot de categorie et canal (pd.get_dummies, dtype=int)
colonnes = ['montant_std', 'heure', 'nuit', 'etranger', 'categorie', 'canal']
X_train = pd.get_dummies(train[colonnes], columns=['categorie', 'canal'], dtype=int)
X_test = pd.get_dummies(test[colonnes], columns=['categorie', 'canal'], dtype=int)

# Le test doit avoir exactement les mêmes colonnes que l'entraînement
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

y_train = train['fraude']
y_test = test['fraude']
print(X_train.shape, X_test.shape)
X_train.head()

## Questions

Répondez dans les cellules de texte situées sous chaque question.

**Question 1.** Combien de lignes et de colonnes contient le fichier ? Classez chaque colonne : identifiant, temporelle, numérique, catégorielle ou cible.

**Réponse 1.** Le fichier contient **10 040 lignes et 8 colonnes**. `id_transaction` et `id_client` sont des identifiants ; `date_heure` est temporelle ; `montant` est numérique ; `categorie`, `canal` et `pays` sont catégorielles ; `fraude` est la cible.

**Question 2.** Quels problèmes de qualité avez-vous trouvés ? Donnez le nombre pour chacun (valeurs manquantes, doublons, montants négatifs, montants extrêmes).

**Réponse 2.**

- **Valeurs manquantes** : 100 montants (1,0 %) et 200 canaux (2,0 %).
- **Doublons exacts** : 40.
- **Montants négatifs** : 6.
- **Montants extrêmes** : 4 montants égaux à 99 999,99 €, et 3 achats de 3 200 €, 4 500 € et 5 800 € (catégorie voyage).

**Question 3.** Parmi les montants très élevés, lesquels avez-vous traités comme des erreurs et lesquels avez-vous gardés ? Justifiez. Pourquoi imputer par la médiane de la *catégorie* plutôt que par la médiane de tous les montants ?

**Réponse 3.** Les montants négatifs et les 4 montants de 99 999,99 € sont des **erreurs évidentes** (valeur impossible, code de saisie) : on les remplace par une valeur manquante. Les achats de 3 200 €, 4 500 € et 5 800 € en voyage sont **plausibles** : ce sont de vraies anomalies, légitimes (aucun n’est une fraude), qu’on garde et qu’on signale. Supprimer toutes les valeurs extrêmes aurait été une erreur en détection d’anomalies.

On impute par la médiane de la catégorie parce que les montants dépendent fortement de la catégorie : médiane d’environ 193 € pour le voyage contre environ 17 € pour le transport. Une médiane globale (environ 29 €) donnerait des valeurs peu réalistes.

**Question 4.** Que révèlent la moyenne et la médiane du montant ? Pourquoi utiliser l’échelle logarithmique ? Quel est le taux de fraude global et que cela implique-t-il pour l’évaluation d’un modèle ?

**Réponse 4.** La moyenne (environ 57 €) est presque le double de la médiane (environ 29 €) : la distribution est **asymétrique à droite**, avec quelques montants très élevés. L’échelle linéaire écrase presque tout près de zéro, l’échelle logarithmique montre la forme de la distribution. Le taux de fraude global est de **1,0 %** (100 fraudes sur 10 000) : les classes sont très déséquilibrées, donc un modèle qui prédit toujours « pas de fraude » aurait 99 % d’accuracy. Il faudra évaluer avec la précision et le rappel.

**Question 5.** Quelles variables semblent les plus liées à la fraude ? Appuyez-vous sur les taux calculés. Cela prouve-t-il que la variable *cause* la fraude ?

**Réponse 5.**

| Variable | Taux de fraude |
|---|---|
| Nuit (heure < 6) | 10,0 % (86 sur 861) contre 0,15 % le jour |
| Canal | en ligne 2,9 % ; carte présente 0,3 % ; sans contact 0 % |
| Pays | étranger 6,3 % contre 0,4 % en France ; NG 24,6 %, DE 10,6 %, US 8,2 %, ES 5,1 % |
| Catégorie | voyage 5,1 %, jeux en ligne 4,7 %, électronique 3,8 %, les autres 0 % |

L’heure, le canal, le pays et la catégorie sont tous informatifs. Il s’agit d’**associations**, pas de causes : ces variables sont aussi liées entre elles (une fraude nocturne est souvent en ligne et à l’étranger), et dans ces données simulées, c’est nous qui avons construit ces liens. Un taux élevé signale un risque, il ne prouve pas une cause.

**Question 6.** Pourquoi calcule-t-on la moyenne et l’écart-type sur l’entraînement uniquement ? Pourquoi la moyenne et l’écart-type standardisés du test ne valent-ils pas exactement 0 et 1 ? Pourquoi découper dans le temps plutôt qu’au hasard ?

**Réponse 6.** Si on calculait μ et σ sur tout le jeu de données, des informations du test passeraient dans l’entraînement : c’est une **fuite de données**, et les performances mesurées seraient trop optimistes. En production, on ne connaît pas le futur : on utilise les paramètres appris sur le passé.

Ici, μ = 56,70 € et σ = 99,24 € sur l’entraînement (6 677 lignes, 64 fraudes). Sur le test (3 323 lignes, 36 fraudes), la moyenne standardisée vaut environ 0,015 et l’écart-type environ 1,27 : le test a ses propres montants, un peu plus dispersés, et c’est normal.

On découpe dans le temps parce que, en production, on apprend sur le passé pour prédire le futur, et les comportements des fraudeurs évoluent. Un découpage au hasard mélange passé et futur et donne une estimation trop favorable.

## Pour aller plus loin (facultatif)

1. Remplacez la standardisation manuelle par `StandardScaler` de scikit-learn (`fit` sur l’entraînement, `transform` sur l’entraînement et le test). Retrouvez-vous les mêmes valeurs ?
2. Appliquez `np.log1p` au montant avant la standardisation. Comment change l’histogramme ?
3. Calculez, pour chaque client, son nombre de transactions et son montant moyen. Les clients ayant subi une fraude se distinguent-ils des autres ?

## Pistes pour « Pour aller plus loin »

- `StandardScaler` : `sc = StandardScaler().fit(train[['montant']])` puis `sc.transform(...)`. Les valeurs sont identiques à une petite différence près : `StandardScaler` utilise l’écart-type de population (division par n) alors que `.std()` de pandas divise par n − 1.
- Après `np.log1p`, l’histogramme du montant est beaucoup plus symétrique, presque en forme de cloche : les variables asymétriques se prêtent mieux aux méthodes qui supposent une distribution proche de la normale (Z-score, séance 3).
- `df.groupby('id_client').agg(nb=('montant', 'count'), moyenne=('montant', 'mean'))` : c’est le début du *feature engineering* par client, qu’on développera en séance 4.